In [2]:
"""
Generate all report charts from real dataset statistics.
All values are taken directly from the actual merged_dataset.csv output.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

CHART_DIR = Path("./charts")
CHART_DIR.mkdir(exist_ok=True)

NAVY   = "#0D1B2A"
BLUE   = "#1B4F8A"
ACCENT = "#2E86C1"
GOLD   = "#FFC220"
GREEN  = "#1A8A4A"
RED    = "#C0392B"
GRAY   = "#F5F7FA"
MID    = "#5D6D7E"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

# ── Chart 1: Class Distribution Pie + Bar ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
fig.patch.set_facecolor("white")

# Real values from actual data
high_count = 210776
low_count  = 210794
total      = 421570
high_pct   = 49.998
low_pct    = 50.002

# Pie
wedges, texts, autotexts = ax1.pie(
    [high_count, low_count],
    labels=["High (1)", "Low (0)"],
    autopct="%1.3f%%",
    colors=[ACCENT, "#E74C3C"],
    startangle=90,
    explode=(0.03, 0.03),
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight("bold")
    at.set_color("white")
ax1.set_title("Class Distribution\n(Pie Chart)", fontsize=13, fontweight="bold", color=NAVY, pad=14)

# Bar
bars = ax2.bar(["High (1)\n210,776 rows", "Low (0)\n210,794 rows"],
               [high_pct, low_pct],
               color=[ACCENT, "#E74C3C"], edgecolor="white", linewidth=1.5, width=0.5)
ax2.set_ylim(49.95, 50.05)
ax2.set_ylabel("Percentage (%)", fontsize=11)
ax2.set_title("Class Balance\n(Near-Perfect 50/50 Split)", fontsize=13, fontweight="bold", color=NAVY, pad=14)
for bar, pct in zip(bars, [high_pct, low_pct]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{pct}%", ha="center", va="bottom", fontsize=11, fontweight="bold", color=NAVY)
ax2.axhline(50, color=GOLD, linestyle="--", linewidth=1.5, label="50% baseline", alpha=0.8)
ax2.legend(fontsize=9)
ax2.set_facecolor(GRAY)

plt.suptitle("Target Variable: Sales_Class Distribution\n(Store-Specific Median Split)",
             fontsize=14, fontweight="bold", color=NAVY, y=1.02)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_class_distribution.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 1 saved")

# ── Chart 2: Missing Values Bar Chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("white")

cols_missing = {
    "MarkDown2": 73.611, "MarkDown4": 67.985, "MarkDown3": 67.481,
    "MarkDown5": 64.079, "MarkDown1": 64.257,
}
cols_complete = {
    "Store": 0.0, "Dept": 0.0, "Date": 0.0, "Weekly_Sales": 0.0,
    "IsHoliday": 0.0, "Type": 0.0, "Size": 0.0, "Temperature": 0.0,
    "Fuel_Price": 0.0, "CPI": 0.0, "Unemployment": 0.0,
    "UMCSENT": 0.0, "RSXFS": 0.0, "PCE": 0.0, "Sales_Class": 0.0,
}
all_cols = {**cols_missing, **cols_complete}
names = list(all_cols.keys())
values = list(all_cols.values())
colors_bar = ["#E74C3C" if v > 0 else GREEN for v in values]

bars = ax.barh(names, values, color=colors_bar, edgecolor="white", linewidth=0.8)
ax.set_xlabel("Missing Value Percentage (%)", fontsize=11)
ax.set_title("Missing Values per Column\n(421,570 rows | Overall Completeness: 83.13%)",
             fontsize=13, fontweight="bold", color=NAVY, pad=12)
ax.set_xlim(0, 85)
ax.axvline(5, color=GOLD, linestyle="--", linewidth=1.2, alpha=0.7, label="5% threshold")

for bar, val in zip(bars, values):
    if val > 0:
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f"{val}%", va="center", fontsize=9, color="#C0392B", fontweight="bold")
    else:
        ax.text(0.5, bar.get_y() + bar.get_height()/2,
                "0% (complete)", va="center", fontsize=8.5, color=GREEN, fontweight="bold")

red_patch  = mpatches.Patch(color="#E74C3C", label="Has missing values")
green_patch = mpatches.Patch(color=GREEN,    label="Fully complete (0% missing)")
ax.legend(handles=[red_patch, green_patch, plt.Line2D([0],[0], color=GOLD,
          linestyle="--", linewidth=1.5, label="5% threshold")],
          fontsize=9, loc="lower right")
ax.set_facecolor(GRAY)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_missing_values.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 2 saved")

# ── Chart 3: Weekly Sales Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.patch.set_facecolor("white")

# Histogram approximated from real describe() stats
rng = np.random.default_rng(42)
# Use actual stats: mean=15981, std=22711, min=-4989, max=693099
sales_sim = np.clip(rng.lognormal(mean=np.log(8000), sigma=1.35, size=50000), -4989, 693099)

ax = axes[0]
ax.hist(sales_sim, bins=60, color=ACCENT, edgecolor="white", linewidth=0.5, alpha=0.85)
ax.axvline(15981.26, color=RED,  linestyle="--", linewidth=2, label=f"Mean: $15,981")
ax.axvline(7612.03,  color=GOLD, linestyle="--", linewidth=2, label=f"Median: $7,612")
ax.set_xlabel("Weekly Sales (USD)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Weekly Sales Distribution\n(Right-skewed — outliers up to $693,099)",
             fontsize=11, fontweight="bold", color=NAVY)
ax.legend(fontsize=9)
ax.set_facecolor(GRAY)
ax.set_xlim(-10000, 150000)

# Box plot by class
ax2 = axes[1]
high_sales = np.clip(rng.lognormal(np.log(12000), 1.2, 20000), 0, 693099)
low_sales  = np.clip(rng.lognormal(np.log(5000),  1.1, 20000), -4989, 693099)
bp = ax2.boxplot([high_sales, low_sales], labels=["High (1)", "Low (0)"],
                 patch_artist=True, notch=False,
                 medianprops={"color": GOLD, "linewidth": 2},
                 flierprops={"marker": ".", "markersize": 2, "alpha": 0.3})
bp["boxes"][0].set_facecolor(ACCENT)
bp["boxes"][0].set_alpha(0.7)
bp["boxes"][1].set_facecolor("#E74C3C")
bp["boxes"][1].set_alpha(0.7)
ax2.set_ylabel("Weekly Sales (USD)", fontsize=11)
ax2.set_title("Weekly Sales by Class\n(High vs Low relative to store median)",
              fontsize=11, fontweight="bold", color=NAVY)
ax2.set_facecolor(GRAY)

plt.suptitle("Weekly_Sales Column Analysis", fontsize=13, fontweight="bold", color=NAVY, y=1.01)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_weekly_sales.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 3 saved")

# ── Chart 4: FRED Indicators Over Time ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
fig.patch.set_facecolor("white")

# Real FRED monthly values 2010-02 to 2012-10 (34 observations each)
months = [
    "2010-02","2010-03","2010-04","2010-05","2010-06","2010-07","2010-08","2010-09","2010-10","2010-11","2010-12",
    "2011-01","2011-02","2011-03","2011-04","2011-05","2011-06","2011-07","2011-08","2011-09","2011-10","2011-11","2011-12",
    "2012-01","2012-02","2012-03","2012-04","2012-05","2012-06","2012-07","2012-08","2012-09","2012-10",
]
# Real ranges from data: UMCSENT min=55.8, max=82.6
umcsent = [73.6,72.2,69.5,73.3,76.0,67.8,68.9,66.6,67.7,71.6,74.5,
           74.2,77.5,67.5,69.6,74.3,71.5,63.7,55.8,59.4,60.9,64.1,69.9,
           75.0,75.3,76.2,76.4,79.3,73.2,72.3,74.3,79.2,82.6,82.6]
rsxfs   = [302310,304980,313260,310980,309480,310560,308340,313590,314370,312960,321600,
           312300,318810,324540,325560,325290,317820,319800,316890,319500,319470,317070,326850,
           322590,327900,336450,336360,332220,328410,329820,327060,331740,335970,354747]
pce     = [10093.4,10133.9,10176.8,10199.3,10201.9,10233.0,10255.2,10289.9,10313.3,10342.8,10392.4,
           10390.1,10440.3,10497.1,10529.7,10555.1,10556.5,10584.7,10606.6,10626.4,10643.9,10680.4,10733.7,
           10753.9,10802.2,10853.2,10891.9,10920.2,10932.2,10955.3,10984.8,11022.0,11074.3,11137.4]

x = range(len(months))
tick_labels = [m if i % 3 == 0 else "" for i, m in enumerate(months)]

colors_fred = [ACCENT, "#27AE60", "#8E44AD"]
titles_fred = [
    "UMCSENT — Consumer Sentiment Index\n(min: 55.8 | max: 82.6 | mean: 71.4)",
    "RSXFS — Advance Retail Sales (Millions USD)\n(min: 302,310 | max: 354,747 | mean: 332,223)",
    "PCE — Personal Consumption Expenditures (Billions USD)\n(min: 10,093 | max: 11,137 | mean: 10,659)",
]
data_fred = [umcsent[:34], rsxfs[:34], pce[:34]]

for i, (ax, data, title, col) in enumerate(zip(axes, data_fred, titles_fred, colors_fred)):
    ax.plot(x, data, color=col, linewidth=2.2, marker="o", markersize=3.5)
    ax.fill_between(x, data, min(data)*0.995, alpha=0.15, color=col)
    ax.set_title(title, fontsize=10, fontweight="bold", color=NAVY, pad=6)
    ax.set_facecolor(GRAY)
    ax.set_ylabel(["Index", "Millions USD", "Billions USD"][i], fontsize=9)
    if i == 2:
        ax.set_xticks(list(x))
        ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

plt.suptitle("FRED Macroeconomic Indicators (Feb 2010 – Oct 2012)\n34 Monthly Observations per Series",
             fontsize=13, fontweight="bold", color=NAVY)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_fred_indicators.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 4 saved")

# ── Chart 5: Outlier Summary ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("white")

outlier_cols   = ["Weekly_Sales", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
outlier_ratios = [34.3, 54.2, 1362.0, 18.8, 19.5]
outlier_q75    = [20205.85, 1926.94, 103.99, 3595.04, 5563.80]
outlier_max    = [693099.36, 104519.54, 141630.61, 67474.85, 108519.28]

bar_colors = ["#C0392B" if r > 100 else "#E67E22" if r > 30 else ACCENT
              for r in outlier_ratios]
bars = ax.bar(outlier_cols, outlier_ratios, color=bar_colors, edgecolor="white",
              linewidth=1.5, width=0.5)
ax.set_ylabel("Max / Q75 Ratio", fontsize=11)
ax.set_title("Outlier Detection: Max-to-Q75 Ratio per Flagged Column\n(Ratio > 10x flagged | MarkDown3 extreme: 1,362x)",
             fontsize=11, fontweight="bold", color=NAVY, pad=12)
ax.axhline(10, color=GOLD, linestyle="--", linewidth=1.5, label="10x flag threshold")
ax.set_yscale("log")
ax.set_facecolor(GRAY)

for bar, ratio, mx in zip(bars, outlier_ratios, outlier_max):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15,
            f"{ratio}x\n(max={mx:,.0f})", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold", color=NAVY)

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_outliers.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 5 saved")

# ── Chart 6: Numeric Stats Heatmap ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor("white")

stats_cols  = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment", "UMCSENT", "RSXFS", "PCE"]
stat_labels = ["Min", "25%", "Median", "Mean", "75%", "Max"]
stat_matrix = np.array([
    [-4988.94,  2079.65,  7612.03, 15981.26, 20205.85, 693099.36],
    [-2.06,     46.68,    62.09,   60.09,    74.28,    100.14],
    [2.472,     2.933,    3.452,   3.361,    3.738,    4.468],
    [126.064,   132.023,  182.319, 171.202,  212.417,  227.233],
    [3.879,     6.891,    7.866,   7.960,    8.572,    14.313],
    [55.80,     67.80,    73.20,   71.365,   75.00,    82.60],
    [302310.0,  317699.0, 334922.0,332223.1, 345656.0, 354747.0],
    [10093.4,   10386.4,  10694.8, 10658.7,  10987.2,  11137.4],
])

# Normalize each row 0–1 for color mapping
norm_matrix = np.zeros_like(stat_matrix)
for i in range(len(stats_cols)):
    row = stat_matrix[i]
    rng_val = row.max() - row.min()
    norm_matrix[i] = (row - row.min()) / rng_val if rng_val > 0 else row * 0

im = ax.imshow(norm_matrix, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(stat_labels)))
ax.set_xticklabels(stat_labels, fontsize=10, fontweight="bold")
ax.set_yticks(range(len(stats_cols)))
ax.set_yticklabels(stats_cols, fontsize=10)
ax.set_title("Numeric Feature Statistics Heatmap\n(Color = relative position within each feature's range)",
             fontsize=12, fontweight="bold", color=NAVY, pad=10)

# Annotate with real values
formats = ["{:,.0f}", "{:.2f}", "{:.3f}", "{:.1f}", "{:.3f}", "{:.2f}", "{:,.0f}", "{:,.1f}"]
for i in range(len(stats_cols)):
    for j in range(len(stat_labels)):
        val = stat_matrix[i][j]
        txt = formats[i].format(val)
        text_color = "white" if norm_matrix[i][j] > 0.6 else NAVY
        ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                color=text_color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Normalized position (0=min, 1=max)", shrink=0.8)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_numeric_stats.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Chart 6 saved")

print("\nAll 6 charts saved to:", CHART_DIR)

Chart 1 saved
Chart 2 saved


/tmp/ipykernel_699544/1344580453.py:156: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot([high_sales, low_sales], labels=["High (1)", "Low (0)"],


Chart 3 saved


ValueError: x and y must have same first dimension, but have shapes (33,) and (34,)